In [1]:
#Evaluate bilingual search performance
'''Using 4 different methods to evaluate:
1. Avg Similarity Score > Proves results are relevant
2. EN vs FR Gap > Proves bilingual balance
3. Cross-lingual ConsistencyProves EN/FR searches agree
4. Score DistributionShows result confidence visually '''

'Using 4 different methods to evaluate:\n1. Avg Similarity Score > Proves results are relevant\n2. EN vs FR Gap > Proves bilingual balance\n3. Cross-lingual ConsistencyProves EN/FR searches agree\n4. Score DistributionShows result confidence visually '

In [2]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
import random
import matplotlib.pyplot as plt

In [3]:
data_folder = "D:/Harshita Ajmani/Code_harshu/NLP/data/"
model = "intfloat/multilingual-e5-base"
 
TOP_K = 5 
MIN_SCORE = 0.7

random.seed(42)
np.random.seed(42)
#num_query = 20 


In [4]:
# Loading and Building Index

embeddings_en = np.load(f"{data_folder}/embeddings_en.npy").astype('float32')
embeddings_fr = np.load(f"{data_folder}/embeddings_fr.npy").astype('float32')

dataframe = pd.read_csv(f"{data_folder}/index_data.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer(model, device=device)

# Build FAISS indexes
dim      = embeddings_en.shape[1]
index_en = faiss.IndexFlatIP(dim)
index_fr = faiss.IndexFlatIP(dim)
index_en.add(embeddings_en)
index_fr.add(embeddings_fr)

print(f"{len(dataframe):,} records indexed")

#print(dim)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


46,468 records indexed


In [7]:
'''Data Quality Filtering: Keep records with 5+ word titles in both languages, and where EN/FR titles differ (to avoid duplicates).'''

filtered_records = dataframe[
    (dataframe['title_en'].str.split().str.len() >= 5) &
    (dataframe['title_fr'].str.split().str.len() >= 5) &
    (dataframe['title_en'] != dataframe['title_fr'])
].copy().reset_index()
print(f"✅ {len(filtered_records):,} records in total")

'''Extracting meaningful keywords from titles by removing common stop words and short words. This helps create more effective search queries.'''

wr_eng = {'the','a','an','of','to','and','in','for','with','by','on','at','from','is','are','was','its','this','that'}
wr_fr = {'le','la','les','de','du','des','et','en','un','une','au','aux','par','sur','pour','dans','ce','qui','que'}

def extract_words(title, lang, n=6):
    """Extract n meaningful consecutive words skipping stop words."""
    stops = wr_eng if lang == 'en' else wr_fr
    words = [w for w in title.split() if w.lower() not in stops and len(w) > 3]
    if len(words) < 2:
        return None
    return " ".join(words[:n])


✅ 30,287 records in total


In [8]:
'''Generating 100 test queries: 
25 monolingual EN, 
25 monolingual FR, 
25 cross-lingual EN→FR, 
and 25 cross-lingual FR→EN. 
Each query is derived from the title of a randomly sampled record that passed the quality filters.'''

print("\nGenerating 100 queries...")

def build_queries(records, query_lang, search_lang, n):
    sampled = records.sample(n=n, random_state=42)
    queries = []
    for _, row in sampled.iterrows():
        query = extract_words(row[f'title_{query_lang}'], query_lang)
        if not query:
            continue
        queries.append({
            "query":      query,
            "query_lang": query_lang,
            "search_idx": search_lang,
            "query_type": f"{query_lang}→{search_lang}",
            "true_idx":   row['index'],
            "title_en":   row['title_en'],
            "title_fr":   row['title_fr'],
        })
    return queries

all_queries = (
    build_queries(filtered_records, 'en', 'en', n=25) +  # monolingual EN
    build_queries(filtered_records, 'fr', 'fr', n=25) +  # monolingual FR
    build_queries(filtered_records, 'en', 'fr', n=25) +  # cross-lingual EN→FR
    build_queries(filtered_records, 'fr', 'en', n=25)    # cross-lingual FR→EN
)
print(f"Generated {len(all_queries)} queries")

'''print("\nSample Queries:")
for q in all_queries[29:36]:
    print(f"{q['query_type']}: '{q['query']}' (True: '{q['title_en']}' / '{q['title_fr']}')")   '''    


Generating 100 queries...
Generated 100 queries


'print("\nSample Queries:")\nfor q in all_queries[29:36]:\n    print(f"{q[\'query_type\']}: \'{q[\'query\']}\' (True: \'{q[\'title_en\']}\' / \'{q[\'title_fr\']}\')")   '

In [9]:
# Search Function

def search(query, search_idx, top_k=5):
    vec = model.encode( f"query: {query}", normalize_embeddings =True ).astype('float32').reshape(1, -1)
    index = index_en if search_idx == 'en' else index_fr
    scores, indices = index.search(vec, top_k)
    
    return indices[0].tolist(), scores[0].tolist()

In [14]:
'''Executing the 100 queries and collecting results, including top similarity scores, worst scores, and the gap between them. This will help evaluate both relevance and bilingual balance of the search results.'''

print(f"\nRunning {len(all_queries)} queries...")

results = []
for i, q in enumerate(all_queries):
    if (i + 1) % 25 == 0:
        print(f"   {i+1}/{len(all_queries)} done...")

    top_indices, top_scores = search(q['query'], q['search_idx'], TOP_K)

    results.append({
        "query":      q['query'],
        "query_type": q['query_type'],
        "search_idx": q['search_idx'],
        "true_idx":   q['true_idx'],
        "top_score":  round(top_scores[0], 4),
        "worst_score":  round(top_scores[-1], 4),
        "gap":        round(top_scores[0] - top_scores[-1], 4),
        "all_scores": top_scores,
        "top_indices": top_indices,
    })

results_df = pd.DataFrame(results)


#print("\nSample Results:")
#print(results_df.sample(10, random_state=42)[['query', 'query_type', 'top_score', 'worst_score', 'gap', 'all_scores']]) 


Running 100 queries...
   25/100 done...
   50/100 done...
   75/100 done...
   100/100 done...


In [15]:
'''Evaluating the results using the Average Similarity Score metric. 
For each query type, 
we calculate the average top similarity score and count how many queries had a top score above the defined threshold (MIN_SCORE). 
We also compute the overall average score across all queries to determine if the search performance meets our expectations.'''

print(f" METRIC 1 — Average Similarity Score")

for qt in ['en→en', 'fr→fr', 'en→fr', 'fr→en']:
    subset   = results_df[results_df['query_type'] == qt]
    avg      = subset['top_score'].mean()
    above    = (subset['top_score'] >= MIN_SCORE).sum()
    status   = "ok" if avg >= MIN_SCORE else "Not ok"
    print(f"  {status} {qt:8} | Avg: {avg:.4f} | Above {MIN_SCORE}: {above}/{len(subset)}")

overall_avg = results_df['top_score'].mean()
print(f"\n  Overall avg score: {overall_avg:.4f}")
print(f"  {'PASS' if overall_avg >= MIN_SCORE else 'FAIL'} — threshold: {MIN_SCORE}")

 METRIC 1 — Average Similarity Score
  ok en→en    | Avg: 0.8860 | Above 0.7: 25/25
  ok fr→fr    | Avg: 0.8758 | Above 0.7: 25/25
  ok en→fr    | Avg: 0.8462 | Above 0.7: 25/25
  ok fr→en    | Avg: 0.8450 | Above 0.7: 25/25

  Overall avg score: 0.8633
  PASS — threshold: 0.7


In [13]:
'''Evaluating the bilingual balance by comparing the average top similarity scores of English and French searches.'''

print(f"METRIC 2 — EN vs FR Score Gap")

en_avg  = results_df[results_df['search_idx'] == 'en']['top_score'].mean()
fr_avg  = results_df[results_df['search_idx'] == 'fr']['top_score'].mean()
gap     = abs(en_avg - fr_avg)

if en_avg < MIN_SCORE or fr_avg < MIN_SCORE:
    print(f" WARNING: Scores too low — gap metric not meaningful!")
else:
    print(f"  EN avg score : {en_avg:.4f}")
    print(f"  FR avg score : {fr_avg:.4f}")
    print(f"  Gap          : {gap:.4f}")
    if gap < 0.01:
        print(f"  Balance      : EXCELLENT")
    elif gap < 0.05:
        print(f"  Balance      : GOOD")
    elif gap < 0.10:
        print(f"  Balance      : ACCEPTABLE")
    else:
        print(f"  Balance      : POOR — consider French fine-tuning")

METRIC 2 — EN vs FR Score Gap
  EN avg score : 0.8655
  FR avg score : 0.8610
  Gap          : 0.0045
  Balance      : EXCELLENT


In [ ]:
'''Evaluating cross-lingual consistency by checking how often the top search results for the same concept in English and French overlap. 
'''

print(f"METRIC 3 — Cross-lingual Consistency")


# For each record search same concept in EN and FR
# Measure overlap in top 5 results
consistency_scores = []

en_queries = [q for q in all_queries if q['query_type'] == 'en→en']
fr_queries = [q for q in all_queries if q['query_type'] == 'fr→fr']

# Match by true_idx — same record searched in both languages
en_dict = {q['true_idx']: q for q in en_queries}
fr_dict = {q['true_idx']: q for q in fr_queries}
common  = set(en_dict.keys()) & set(fr_dict.keys())

print(f"  Matched {len(common)} record pairs for consistency check")

for idx in common:
    en_indices, _ = search(en_dict[idx]['query'], 'en', TOP_K)
    fr_indices, _ = search(fr_dict[idx]['query'], 'fr', TOP_K)
    overlap = len(set(en_indices) & set(fr_indices))
    consistency_scores.append(overlap / TOP_K)

avg_consistency = np.mean(consistency_scores) * 100
print(f"  Avg result overlap  : {avg_consistency:.1f}%")
if avg_consistency >= 60:
    print(f"  Consistency         : ✅ STRONG")
elif avg_consistency >= 40:
    print(f"  Consistency         : ⚠️  MODERATE")
else:
    print(f"  Consistency         : ❌ WEAK")


METRIC 3 — Cross-lingual Consistency
  Matched 25 record pairs for consistency check
  Avg result overlap  : 58.4%
  Consistency         : ⚠️  MODERATE


In [18]:
'''print(f" METRIC 4 — Score Distribution")

for qt in ['en→en', 'fr→fr', 'en→fr', 'fr→en']:
    subset = results_df[results_df['query_type'] == qt]['top_score']
    print(f"  {qt:8} | min: {subset.min():.3f} | max: {subset.max():.3f} | std: {subset.std():.4f}")

print(f"\n  Low std  = consistent scores = reliable")
print(f"  High std = scattered scores  = unpredictable")'''

'print(f" METRIC 4 — Score Distribution")\n\nfor qt in [\'en→en\', \'fr→fr\', \'en→fr\', \'fr→en\']:\n    subset = results_df[results_df[\'query_type\'] == qt][\'top_score\']\n    print(f"  {qt:8} | min: {subset.min():.3f} | max: {subset.max():.3f} | std: {subset.std():.4f}")\n\nprint(f"\n  Low std  = consistent scores = reliable")\nprint(f"  High std = scattered scores  = unpredictable")'

In [21]:
'''fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Semantic Search Evaluation — 4 Metrics", fontsize=14, fontweight='bold')

# Plot 1 — Avg score per query type (Metric 1)
query_types = ['en→en', 'fr→fr', 'en→fr', 'fr→en']
avg_scores  = [results_df[results_df['query_type'] == qt]['top_score'].mean() for qt in query_types]
colors      = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
bars = axes[0,0].bar(query_types, avg_scores, color=colors)
axes[0,0].set_title("Metric 1 — Avg Similarity Score by Query Type")
axes[0,0].set_ylabel("Avg Score")
axes[0,0].set_ylim(0.75, 0.95)
axes[0,0].axhline(y=MIN_SCORE, color='red', linestyle='--', alpha=0.5, label=f'Min threshold ({MIN_SCORE})')
axes[0,0].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[0,0].legend(fontsize=8)


# Plot 2 — EN vs FR gap (Metric 2)
axes[0,1].bar(['EN Index', 'FR Index'], [en_avg, fr_avg], color=['#2196F3', '#4CAF50'], width=0.4)
axes[0,1].set_title(f"Metric 2 — EN vs FR Score Gap ({gap:.4f})")
axes[0,1].set_ylabel("Avg Score")
axes[0,1].set_ylim(0.75, 0.95)
for i, v in enumerate([en_avg, fr_avg]):
    axes[0,1].text(i, v + 0.001, f"{v:.4f}", ha='center', fontsize=11, fontweight='bold')

# Plot 3 — Cross-lingual consistency (Metric 3)
axes[1,0].hist(consistency_scores, bins=6, color='#9C27B0', edgecolor='white')
axes[1,0].set_title(f"Metric 3 — Cross-lingual Consistency (avg: {avg_consistency:.1f}%)")
axes[1,0].set_xlabel("Overlap Ratio (0=none, 1=identical)")
axes[1,0].set_ylabel("Count")
axes[1,0].axvline(x=np.mean(consistency_scores), color='red', linestyle='--', label=f'Mean: {np.mean(consistency_scores):.2f}')
axes[1,0].legend()

# Plot 4 — Score distribution (Metric 4)
for qt, color in zip(query_types, colors):
    scores = results_df[results_df['query_type'] == qt]['top_score']
    axes[1,1].hist(scores, bins=12, alpha=0.5, color=color, label=qt)
axes[1,1].set_title("Metric 4 — Score Distribution by Query Type")
axes[1,1].set_xlabel("Similarity Score")
axes[1,1].set_ylabel("Count")
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR + "evaluation_results.png", dpi=150, bbox_inches='tight')
plt.show()'''



'fig, axes = plt.subplots(2, 2, figsize=(14, 10))\nfig.suptitle("Semantic Search Evaluation — 4 Metrics", fontsize=14, fontweight=\'bold\')\n\n# Plot 1 — Avg score per query type (Metric 1)\nquery_types = [\'en→en\', \'fr→fr\', \'en→fr\', \'fr→en\']\navg_scores  = [results_df[results_df[\'query_type\'] == qt][\'top_score\'].mean() for qt in query_types]\ncolors      = [\'#2196F3\', \'#4CAF50\', \'#FF9800\', \'#E91E63\']\nbars = axes[0,0].bar(query_types, avg_scores, color=colors)\naxes[0,0].set_title("Metric 1 — Avg Similarity Score by Query Type")\naxes[0,0].set_ylabel("Avg Score")\naxes[0,0].set_ylim(0.75, 0.95)\naxes[0,0].axhline(y=MIN_SCORE, color=\'red\', linestyle=\'--\', alpha=0.5, label=f\'Min threshold ({MIN_SCORE})\')\naxes[0,0].bar_label(bars, fmt=\'%.4f\', padding=3, fontsize=9)\naxes[0,0].legend(fontsize=8)\n\n\n# Plot 2 — EN vs FR gap (Metric 2)\naxes[0,1].bar([\'EN Index\', \'FR Index\'], [en_avg, fr_avg], color=[\'#2196F3\', \'#4CAF50\'], width=0.4)\naxes[0,1].set_title

In [23]:
results_df.to_csv(data_folder + "evaluation_results.csv", index=False)